<a href="https://colab.research.google.com/github/thinus283-ux/LR/blob/main/Full_real_SPARC_topological_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# =============================================
# FULL 175 SPARC GALAXIES - Topological vs NFW Comparison
# Uses your exact working data loader from GitHub
# =============================================

!pip install numpy scipy matplotlib pandas -q

import numpy as np
import pandas as pd
import zipfile
import os
import warnings
warnings.filterwarnings("ignore")

# ================== DOWNLOAD & LOAD REAL SPARC DATA ==================
print("Downloading official Rotmod_LTG.zip ...")
!wget -q https://astroweb.case.edu/SPARC/Rotmod_LTG.zip -O Rotmod_LTG.zip

with zipfile.ZipFile("Rotmod_LTG.zip", 'r') as zip_ref:
    zip_ref.extractall("sparc_data")

dat_files = [f for f in os.listdir("sparc_data") if f.endswith('.dat')]
print(f"Processing {len(dat_files)} real SPARC galaxies...\n")

# ================== PARAMETERS ==================
alpha = 12.0
kappa = 2.8
G = 4.30091e-3
topo_norm = 48.0

# ================== CORE FUNCTIONS ==================
def compute_vorticity_current(r, v_bary, alpha, kappa):
    omega = v_bary / r
    domega_dr = np.gradient(omega, r, edge_order=2)
    F = np.abs(domega_dr)
    J = alpha * F * (1 + kappa * np.exp(-r/18))
    return J

def topological_stress_density(r, J, kappa, topo_norm):
    div_J = np.gradient(J, r, edge_order=2)
    rho_topo = topo_norm * kappa * np.abs(div_J) * np.exp(-r/32) / (1 + r/25)
    return rho_topo

def topo_acceleration(r, rho_topo, G):
    M_enc = np.cumsum(4 * np.pi * r**2 * rho_topo) * np.gradient(r)
    a = G * M_enc / (r**2 + 1e-8)
    return a

def nfw_acceleration(r, v_bary, G=4.30091e-3):
    v_peak = np.max(v_bary)
    M200 = 1e10 * (v_peak / 100)**4
    c = 10
    Rs = r / (c * (np.log(1 + c) - c / (1 + c)))
    x = r / Rs
    return G * M200 * (np.log(1 + x) - x / (1 + x)) / (r**2 * (np.log(1 + c) - c / (1 + c)))

def process_galaxy(r, Vobs, Vgas, Vdisk, Vbul, model='topo'):
    Vbary = np.sqrt(np.maximum(Vgas**2 + Vdisk**2 + Vbul**2, 0))

    if model == 'topo':
        J = compute_vorticity_current(r, Vbary, alpha, kappa)
        rho_topo = topological_stress_density(r, J, kappa, topo_norm)
        a_extra = topo_acceleration(r, rho_topo, G)
    else:
        a_extra = nfw_acceleration(r, Vbary, G)

    V_total = np.sqrt(np.maximum(Vbary**2 + a_extra * r, 0))
    rms = np.sqrt(np.mean((V_total - Vobs)**2))
    return rms

# ================== RUN BOTH MODELS ==================
results = []

for fname in dat_files:
    path = os.path.join("sparc_data", fname)
    try:
        data = np.loadtxt(path, skiprows=1)
        if data.shape[1] < 6:
            continue

        r     = data[:, 0]
        Vobs  = data[:, 1]
        Vgas  = data[:, 3]
        Vdisk = data[:, 4]
        Vbul  = data[:, 5]

        rms_topo = process_galaxy(r, Vobs, Vgas, Vdisk, Vbul, model='topo')
        rms_nfw  = process_galaxy(r, Vobs, Vgas, Vdisk, Vbul, model='nfw')

        results.append({
            'Galaxy': fname.replace('.dat', ''),
            'RMS_Topo': round(rms_topo, 3),
            'RMS_NFW': round(rms_nfw, 3)
        })
    except:
        continue

df_results = pd.DataFrame(results)

# ================== FINAL STATISTICS ==================
print("\n" + "="*90)
print("TOPOLOGICAL MODEL vs NFW (ΛCDM) — ALL 175 REAL SPARC GALAXIES")
print("="*90)
print(df_results.sort_values('RMS_Topo'))
print("\n" + "-"*90)
print(f"Topological Model - Median RMS : {df_results['RMS_Topo'].median():.3f} km/s")
print(f"NFW (ΛCDM)        - Median RMS : {df_results['RMS_NFW'].median():.3f} km/s")
print(f"Galaxies where Topo beats NFW  : {(df_results['RMS_Topo'] < df_results['RMS_NFW']).sum()}")
print("="*90)

df_results.to_csv('SPARC_Topo_vs_NFW_175_Comparison.csv', index=False)
print("\n✅ Full comparison saved to: SPARC_Topo_vs_NFW_175_Comparison.csv")

Processing 175 real SPARC galaxies...


TOPOLOGICAL MODEL vs NFW (ΛCDM) — ALL 175 REAL SPARC GALAXIES
              Galaxy  RMS_Topo    RMS_NFW
43   UGC05918_rotmod     4.846    273.013
41     F567-2_rotmod     4.949    556.869
114    IC2574_rotmod     5.846    468.398
167    F583-4_rotmod     6.611   1042.516
88   UGC05414_rotmod     6.687    930.807
..               ...       ...        ...
159   NGC2903_rotmod   204.587  38224.369
37   UGC02953_rotmod   204.982  57702.562
128  UGC05253_rotmod   263.061  43402.016
47   UGC06787_rotmod   264.372  54541.313
42   UGC03546_rotmod   393.237  70357.433

[175 rows x 3 columns]

------------------------------------------------------------------------------------------
Topological Model - Median RMS : 34.869 km/s
NFW (ΛCDM)        - Median RMS : 1611.266 km/s
Galaxies where Topo beats NFW  : 175

✅ Full comparison saved to: SPARC_Topo_vs_NFW_175_Comparison.csv
